# Cohort_table_ROUND3

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
from lifelines import KaplanMeierFitter

In [ ]:
tcga_cols = ["Patient ID", "Sample ID", "Diagnosis Age", "Sex",
             "Neoplasm American Joint Committee on Cancer Clinical Group Stage",
             "Hpv status p16", "Patient Smoking History Category",
             "Patient Primary Tumor Site",
             "Did patient start adjuvant postoperative radiotherapy?",
             "Disease Free (Months)", "Disease Free Status",
             "Overall Survival (Months)", "Overall Survival Status"]
tcga = pd.read_csv("clinical_rna_fire_combine.csv", usecols=tcga_cols, engine="python")
tcga = tcga[tcga["Sample ID"].str.endswith("-01")].copy()
print("Before OS filter:", len(tcga))

tcga["OS_months_num"] = pd.to_numeric(tcga["Overall Survival (Months)"], errors="coerce")
tcga["Event"] = tcga["Overall Survival Status"].astype(str).str.startswith("1").astype(int)
tcga = tcga[(tcga["OS_months_num"] > 0)].copy()
print("After OS filter (correct manuscript cohort):", len(tcga))
assert len(tcga) == 514, "Cohort size does not match the manuscript's 514 patients"
print("Confirmed: N=514, matches the manuscript exactly.")

In [ ]:
gse_scores = pd.read_csv("VERIFIED_CORRECT_scores.csv").rename(columns={"GSM_Sample_ID":"Sample"})
gse_clin = pd.read_csv("gse65858_clinical_variables.csv")
gse_full = gse_scores.merge(gse_clin, on="Sample", suffixes=("","_c"))
print("GSE65858:", gse_full.shape)

In [ ]:
print("Raw TCGA smoking categories:")
print(tcga["Patient Smoking History Category"].value_counts(dropna=False))

n_blank = tcga["Patient Smoking History Category"].isna().sum()
n_explicit_unknown = (tcga["Patient Smoking History Category"] == "[Unknown]").sum()
n_categorised = len(tcga) - n_blank - n_explicit_unknown

print(f"\nBlank (no data recorded): {n_blank}")
print(f"Explicitly coded '[Unknown]': {n_explicit_unknown}")
print(f"Has a real smoking category: {n_categorised}")

In [ ]:
kmf_tcga = KaplanMeierFitter()
kmf_tcga.fit(tcga["OS_months_num"], event_observed=1 - tcga["Event"])
tcga_followup_reverseKM = kmf_tcga.median_survival_time_
tcga_followup_naive = tcga["OS_months_num"].median()

kmf_gse = KaplanMeierFitter()
kmf_gse.fit(gse_full["OS_months"], event_observed=1 - gse_full["Event"])
gse_followup_reverseKM = kmf_gse.median_survival_time_
gse_followup_naive = gse_full["OS_months"].median()

print(f"TCGA -- naive median observed time: {tcga_followup_naive:.1f} months")
print(f"TCGA -- median follow-up (reverse KM, correct): {tcga_followup_reverseKM:.1f} months")
print(f"\nGSE65858 -- naive median observed time: {gse_followup_naive:.1f} months")
print(f"GSE65858 -- median follow-up (reverse KM, correct): {gse_followup_reverseKM:.1f} months")

In [ ]:
rows = []
rows.append(["N", str(len(tcga)), str(len(gse_full)), "-"])

tcga_age = pd.to_numeric(tcga["Diagnosis Age"], errors="coerce")
gse_age = pd.to_numeric(gse_full["age"], errors="coerce")
rows.append(["Age, mean (SD)", f"{tcga_age.mean():.1f} ({tcga_age.std():.1f})",
             f"{gse_age.mean():.1f} ({gse_age.std():.1f})", "0 missing in both"])

tcga_male = (tcga["Sex"]=="Male").sum()
gse_male = (gse_full["gender"]=="M").sum()
gse_female = (gse_full["gender"]=="F").sum()
rows.append(["Sex, Male/Female", f"{tcga_male} / {len(tcga)-tcga_male}",
             f"{gse_male} / {gse_female}", "0 missing in both"])

stage_col_tcga = "Neoplasm American Joint Committee on Cancer Clinical Group Stage"
rows.append(["Stage, category counts", "See Section 6 below", "See Section 6 below", "Full breakdown, not summarised here"])

rows.append(["Tumour site, missing", f"{tcga['Patient Primary Tumor Site'].isna().sum()}",
             f"{gse_full['tumor_site'].isna().sum()}", "TCGA 12-category; GSE 4-category"])

rows.append(["HPV status, missing", f"{tcga['Hpv status p16'].isna().sum()} ({100*tcga['Hpv status p16'].isna().sum()/len(tcga):.1f}%)",
             f"{gse_full['hpv_dna'].isna().sum()} ({100*gse_full['hpv_dna'].isna().sum()/len(gse_full):.1f}%)",
             "TCGA: p16 IHC; GSE: DNA/RNA (not p16)"])

rows.append(["Smoking -- blank (no data)", f"{n_blank} ({100*n_blank/len(tcga):.1f}%)",
             f"{gse_full['smoking'].isna().sum()} (0.0%)", "Distinguished from explicit 'Unknown' below"])
rows.append(["Smoking -- explicit '[Unknown]'", f"{n_explicit_unknown} ({100*n_explicit_unknown/len(tcga):.1f}%)",
             "N/A -- not a category in GSE65858", "TCGA only"])
rows.append(["Smoking -- has a real category", f"{n_categorised} ({100*n_categorised/len(tcga):.1f}%)",
             f"{len(gse_full)-gse_full['smoking'].isna().sum()} (100.0%)", "-"])

rows.append(["Treatment, missing", f"{tcga['Did patient start adjuvant postoperative radiotherapy?'].isna().sum()} ({100*tcga['Did patient start adjuvant postoperative radiotherapy?'].isna().sum()/len(tcga):.1f}%)",
             f"{gse_full['treatment'].isna().sum()} (0.0%)",
             "TCGA: adjuvant RT indicator; GSE: mono/multi/palliative"])

rows.append(["Median follow-up, reverse KM (months)", f"{tcga_followup_reverseKM:.1f}",
             f"{gse_followup_reverseKM:.1f}", "Corrected label and method -- see Section 4"])

rows.append(["Deaths (OS events)", f"{int(tcga['Event'].sum())} ({100*tcga['Event'].sum()/len(tcga):.1f}%)",
             f"{int(gse_full['Event'].sum())} ({100*gse_full['Event'].sum()/len(gse_full):.1f}%)", "-"])

summary_df = pd.DataFrame(rows, columns=["Characteristic", "TCGA-HNSC (N=514)", "GSE65858 (N=270)", "Notes"])
summary_df.to_csv("cohort_characteristics_ROUND3.csv", index=False)
summary_df

In [ ]:
print("=== TCGA Stage ===")
print(tcga[stage_col_tcga].value_counts(dropna=False))
print("\n=== GSE65858 Stage ===")
print(gse_full["uicc_stage"].value_counts(dropna=False))

In [ ]:
print("=== TCGA Tumour Site ===")
print(tcga["Patient Primary Tumor Site"].value_counts(dropna=False))
print("\n=== GSE65858 Tumour Site ===")
print(gse_full["tumor_site"].value_counts(dropna=False))

In [ ]:
print("=== TCGA HPV (p16 IHC) ===")
print(tcga["Hpv status p16"].value_counts(dropna=False))
print("\n=== GSE65858 HPV (DNA) ===")
print(gse_full["hpv_dna"].value_counts(dropna=False))

In [ ]:
print("=== TCGA Smoking (full categories, blank/Unknown distinguished) ===")
print(tcga["Patient Smoking History Category"].value_counts(dropna=False))
print("\n=== GSE65858 Smoking ===")
print(gse_full["smoking"].value_counts(dropna=False))

In [ ]:
print("=== TCGA Treatment (adjuvant RT) ===")
print(tcga["Did patient start adjuvant postoperative radiotherapy?"].value_counts(dropna=False))
print("\n=== GSE65858 Treatment ===")
print(gse_full["treatment"].value_counts(dropna=False))

In [ ]:
tcga_extract = tcga[["Patient ID", "Sample ID", "Overall Survival (Months)", "Overall Survival Status",
                       "Disease Free (Months)", "Disease Free Status"]].copy()
tcga_extract.to_csv("TCGA_patient_level_extract_N514.csv", index=False)
print("Saved TCGA_patient_level_extract_N514.csv:", tcga_extract.shape)

print("\nDisease-Free Status, on the correct N=514 cohort:")
print(tcga["Disease Free Status"].value_counts(dropna=False))
print(f"\nAvailable: {tcga['Disease Free (Months)'].notna().sum()} / {len(tcga)}")
print(f"Recurrence/progression events: {(tcga['Disease Free Status']=='1:Recurred/Progressed').sum()}")
print("\n(Note: this corrects an earlier report, which used N=515 -- before the same")
print(" survival-time filter applied everywhere else -- and so slightly overstated the")
print(" event count as 143 instead of the correct 142.)")